## Dimention Reduction Method -Pathway 2
- In this model, we used the dimension reduction 
- It classifies the specialties into super classes, we researched 5 superclasses to be used
- used to Gridsearch to do hyperparameter tuning

### Linear SVC Model

#### Import all the libraries and packages needed

In [26]:
#import required libraries and packages
import pandas as pd
import numpy as np
import re
# Train-test split
from  sklearn.model_selection import train_test_split
#from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
#models
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
#evaluation of the model
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)
#visualization 
import matplotlib.pyplot as plt
# nltk
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

#download stopwords from nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /home/obanda/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### Load and Prepreocess the data

In [27]:

# Load the cleaned dataset and clinical stop words
df = pd.read_csv("../data/super_clinical_mtsamples.csv")
# Load custom clinical stop words
with open("../data/clinical-stopwords.txt") as f:
    clinical_stopwords={
        line.strip().lower()
        for line in f
        if line.strip() and not line.startswith("#")
    }
#lematize and clean up the re
def preprocess_text(text):
    """
    Clean and preprocess clinical text for TF-IDF.
    """

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers and punctuation (keep only letters and spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Split text into individual words
    tokens = word_tokenize(text)

    # Remove stop words and lemmatize the remaining words
    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in clinical_stopwords]

    # Join the cleaned words back into a sentence
    return " ".join(cleaned_tokens)
# Apply preprocessing to all transcriptions
df["cleaned_text"] = df["transcription"].apply(preprocess_text)


In [28]:
df.shape

(4012, 8)

In [29]:

# define the features
X = df.transcription
y = df.clinical_superclass

# split the data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Create TF-IDF instance
tfidf = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1,2)
)

# Learn vocabulary and transform the training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data
X_test_tfidf = tfidf.transform(X_test)

# Train the Linear SVC model
svc_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Evaluate using 5-fold cross validation
scores = cross_validate(
    svc_model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="f1_macro"
)

# print("Fold scores:", scores)
# print(f"SVC Mean Macro F1: {np.mean(scores['test_score']):.2f}")
#train the model
svc_model.fit(X_train_tfidf, y_train)

#time to make predictions
y_pred_svc = svc_model.predict(X_test_tfidf)

#model evaluation
accuracy = accuracy_score(y_test, y_pred_svc)
# print(f"Accuracy: {accuracy:.2f}")
# print(classification_report(y_test, y_pred_svc))


# Define the features

# Create a pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svc", LinearSVC(class_weight="balanced", random_state=42))
])

# Define the hyperparameter grid
param_grid = {
    "tfidf__max_features": [5000, 10000],
    "tfidf__min_df": [2, 5],
    "tfidf__max_df": [0.90, 0.95],
    "tfidf__ngram_range": [(1,1), (1,2)],

    "svc__C": [0.1, 1, 10]
}

# Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Train the pipeline
grid.fit(X_train, y_train)




/home/obanda/eneza/project/Eneza-Data-Science-Residential-training-2026/project_5/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/obanda/eneza/project/Eneza-Data-Science-Residential-training-2026/project_5/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/obanda/eneza/project/Eneza-Data-Science-Residential-training-2026/project_5/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/obanda/eneza/project/Eneza-Data-Science-Residential-training-2026/project_5/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/obanda/ene

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svc__C': [0.1, 1, ...], 'tfidf__max_df': [0.9, 0.95], 'tfidf__max_features': [5000, 10000], 'tfidf__min_df': [2, 5], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"

### Multinormial Logistic Regression Model

In [30]:
#Learn the vocabulary from the training data | Computes IDF weights
X_train_tfidf = tfidf.fit_transform(X_train) #fit and transform X_train
X_test_tfidf = tfidf.transform(X_test) #just transform X_test

#Create the Logistic Regression model
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

#calculate Macro F1 average | cross validation
scores = cross_validate(
    lr_model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="f1_macro"
)

# print("Fold scores: ",scores)
# print(f"LR Mean Macro F1: {np.mean(scores["test_score"]):.2f}")

# Train the model
lr_model.fit(X_train_tfidf, y_train)

# Predict the test set
y_pred_lr = lr_model.predict(X_test_tfidf)

#Time to Evaluate the model
accuracy_lr = accuracy_score(y_test, y_pred_lr)
# print(f"Accuracy: {accuracy_lr:.2f}")
# print(classification_report(y_test, y_pred_lr))
# Define the features
X = df.transcription
y = df.clinical_superclass

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)
# Create a pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000,random_state=42))
])

# Define the hyperparameter grid
param_grid = {
    "tfidf__max_features": [5000, 10000],
    "tfidf__min_df": [2, 5],
    "tfidf__max_df": [0.90, 0.95],
    "tfidf__ngram_range": [(1,1), (1,2)],

    "lr__C": [0.1, 1, 10]
}

# Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Train the pipeline
grid.fit(X_train, y_train)

# Best results
#print("Best Parameters:")
#print(grid.best_params_)

#print("\nBest Cross-Validation Macro F1:")
#print(grid.best_score_)

#get the best model
best_model = grid.best_estimator_
y_pred_lr = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_lr)
#print(f"Accuracy: {accuracy:.2f}")
#print(classification_report(y_test, y_pred_lr))

In [33]:
print("=" * 100)
print("LINEAR SVC - BEFORE HYPERPARAMETER TUNING".center(100))
print("=" * 100)

print("\nCross-Validation Results")
print("-" * 100)
print("Fold Scores:", scores["test_score"])
print(f"Mean Macro F1 : {np.mean(scores['test_score']):.4f}")

print("\nTest Set Performance")
print("-" * 100)
print(f"Accuracy      : {accuracy_score(y_test, y_pred_svc):.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred_svc))


print("\n" + "=" * 100)
print("LINEAR SVC - AFTER HYPERPARAMETER TUNING".center(100))
print("=" * 100)

print("\nBest Hyperparameters")
print("-" * 100)
for param, value in grid.best_params_.items():
    print(f"{param:<25}: {value}")

print(f"\nBest CV Macro F1 : {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest Set Performance")
print("-" * 100)
print(f"Accuracy      : {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))


print("\n\n" + "=" * 100)
print("LOGISTIC REGRESSION - BEFORE HYPERPARAMETER TUNING".center(100))
print("=" * 100)

print("\nCross-Validation Results")
print("-" * 100)
print("Fold Scores:", scores["test_score"])
print(f"Mean Macro F1 : {np.mean(scores['test_score']):.4f}")

print("\nTest Set Performance")
print("-" * 100)
print(f"Accuracy      : {accuracy_lr:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr))


print("\n" + "=" * 100)
print("LOGISTIC REGRESSION - AFTER HYPERPARAMETER TUNING".center(100))
print("=" * 100)

print("\nBest Hyperparameters")
print("-" * 100)
for param, value in grid.best_params_.items():
    print(f"{param:<25}: {value}")

print(f"\nBest CV Macro F1 : {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred_lr = best_model.predict(X_test)

print("\nTest Set Performance")
print("-" * 100)
print(f"Accuracy      : {accuracy_score(y_test, y_pred_lr):.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr))

print("\n" + "=" * 100)
print("END OF MODEL COMPARISON".center(100))
print("=" * 100)

                             LINEAR SVC - BEFORE HYPERPARAMETER TUNING                              

Cross-Validation Results
----------------------------------------------------------------------------------------------------
Fold Scores: [0.58005277 0.5344776  0.5242648  0.55548728 0.59258556]
Mean Macro F1 : 0.5574

Test Set Performance
----------------------------------------------------------------------------------------------------
Accuracy      : 0.5756

Classification Report
                                precision    recall  f1-score   support

      diagnostic_and_pathology       0.16      0.21      0.18        86
internal_and_systemic_medicine       0.51      0.56      0.53       344
     neurology_and_behavioural       0.35      0.44      0.39        75
      rehab_and_applied_health       0.37      0.42      0.39        48
       surgical_and_procedural       0.75      0.66      0.70       651

                      accuracy                           0.58      1204
    